# Plot saved results
Everything here reads only the CSVs in `results/`; no GPU or model is needed.
Every table is in long (tidy) format and has `model` and `run_id` columns.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else str(Path.cwd() / "src"))

import pandas as pd
import matplotlib.pyplot as plt
from crashllm.config import load_config
from crashllm.results import load_table, list_runs
from crashllm import plotting

cfg = load_config()
RD = cfg["paths"]["results_dir"]
list_runs(RD)   # every finished run

## Load tables
`runs="latest"` gives the newest finished run of each model. You can also use `"all"` or a list such as `["BERT_multitask/20261006-153012"]`.

In [ ]:
def load(name, runs="latest", include_debug=False):
    df = load_table(RD, name, runs=runs)
    if not df.empty and not include_debug:
        df = df[~df["model"].str.contains("_DEBUG")]
    return df

overall   = load("overall_metrics")
variables = load("variable_metrics")
per_class = load("per_class_metrics")
confusion = load("confusion_matrices")
cases     = load("case_results")
history   = load("training_history")
overall

## Ready-made figures

In [ ]:
plotting.overall_comparison(overall);

In [ ]:
plotting.metric_by_variable(variables, "macro_f1");

In [ ]:
plotting.fields_correct_distribution(cases);

In [ ]:
model = overall["model"].iloc[0]
plotting.confusion_matrix(confusion, model, "crash_type");

In [ ]:
if not history.empty:
    plotting.training_curves(history[history["level"] == "step"]);

## Custom example: heatmap of models × variables

In [ ]:
pivot = variables.pivot_table(index="model", columns="variable", values="match_percentage")
fig, ax = plt.subplots(figsize=(10, 0.6 * len(pivot) + 2))
im = ax.imshow(pivot.values, cmap="viridis", vmin=0, vmax=100)
ax.set_xticks(range(pivot.shape[1]), pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(pivot.shape[0]), pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.1f}", ha="center", va="center", color="white")
fig.colorbar(im, ax=ax, label="match %")
plt.tight_layout()

## Custom example: per-case errors and LLM latency

In [ ]:
# Crashes that every model got wrong on crash_type
wrong = cases[~cases["correct_crash_type"].astype(bool)]
hard = wrong.groupby("case_id")["model"].nunique()
hard[hard == cases["model"].nunique()].index.tolist()[:20]

In [ ]:
import json
def load_raw(model_name, run_id=None):
    run_id = run_id or (RD / model_name / "LATEST").read_text().strip()
    with open(RD / model_name / run_id / "raw_outputs.jsonl") as f:
        return pd.DataFrame([json.loads(l) for l in f])

# e.g. raw = load_raw("Llama_3.1_8B_QLoRA"); raw["latency_sec"].describe()